In [2]:
import pandas as pd
import random

In [3]:
# Lista de medios de transporte
modos = ['auto', 'moto', 'bus', 'bici']

# Seteamos una semilla y generamos datos de ejemplo en formato ancho
random.seed(2020)

# Construimos el DataFrame
data = pd.DataFrame({
'id': range(1,101),
'tiempo_auto': [random.randint(10, 30) for _ in range(100)],
'tiempo_moto': [random.randint(10, 30) for _ in range(100)],
'tiempo_bus': [random.randint(10, 60) for _ in range(100)],
'tiempo_bici': [random.randint(10, 70) for _ in range(100)],
'modo_elegido': [random.choice(modos) for _ in range(100)]
})

# Extraemos las 2 primeras filas del dataset generado
data.head(2)

,id,tiempo_auto,tiempo_moto,tiempo_bus,tiempo_bici,modo_elegido
0,1,29,25,39,24,moto
1,2,29,29,60,18,bici


### Pasar de modo ancho a modo largo

In [4]:
df=pd.melt(data, id_vars=['id','modo_elegido'], var_name='modo', value_name='tiempo')

In [5]:
df=df.set_index('id')

In [6]:
df.head()

,modo_elegido,modo,tiempo
id,,,
1,moto,tiempo_auto,29
2,bici,tiempo_auto,29
3,auto,tiempo_auto,15
4,auto,tiempo_auto,24
5,moto,tiempo_auto,24


In [7]:
df['modo']=df['modo'].str.replace('tiempo_','')

In [8]:
df

,modo_elegido,modo,tiempo
id,,,
1,moto,auto,29
2,bici,auto,29
3,auto,auto,15
4,auto,auto,24
5,moto,auto,24
...,...,...,...
96,moto,bici,52
97,bici,bici,30
98,bici,bici,30


### Eliminar faltantes

In [9]:
hogares=pd.read_excel('hogares.xlsx')

In [11]:
hogares.head()

,id_propiedad,distrito,barrio,ambientes,precio_usd
0,1,oeste,bella_vista,3,87000.5
1,2,norte,refineria,3,104000.0
2,3,oeste,cinco_esquinas,2,98000.0
3,4,sur,saladillo,2,85000.0
4,5,centro,centro,2,65000.0


In [12]:
hogares.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_propiedad  50 non-null     int64  
 1   distrito      50 non-null     object 
 2   barrio        50 non-null     object 
 3   ambientes     50 non-null     int64  
 4   precio_usd    48 non-null     float64
dtypes: float64(1), int64(2), object(2)
memory usage: 2.1+ KB


In [13]:
hogares.isnull().sum()

id_propiedad    0
distrito        0
barrio          0
ambientes       0
precio_usd      2
dtype: int64

In [16]:
# filtramos datos faltantes en precios_usd
hogares[hogares['precio_usd'].isna()]

,id_propiedad,distrito,barrio,ambientes,precio_usd
10,11,centro,martin,2,NaN
13,14,norte,alberdi,2,NaN


### Rellenamos los NULLS con el promedio

In [17]:
hogares_mean=hogares.copy()

In [18]:
precio_promedio=hogares_mean['precio_usd'].mean()

In [19]:
hogares_mean['precio_usd']=hogares_mean['precio_usd'].fillna(precio_promedio)

In [20]:
hogares_mean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_propiedad  50 non-null     int64  
 1   distrito      50 non-null     object 
 2   barrio        50 non-null     object 
 3   ambientes     50 non-null     int64  
 4   precio_usd    50 non-null     float64
dtypes: float64(1), int64(2), object(2)
memory usage: 2.1+ KB


In [21]:
print(hogares_mean.iloc[[10,13]])

    id_propiedad distrito   barrio  ambientes    precio_usd
10            11   centro   martin          2  84555.052083
13            14    norte  alberdi          2  84555.052083


### Segmentacion de Datos, agrupar

In [37]:
#Promedio por barrio
hogares.groupby('barrio')['precio_usd'].mean().sort_values(ascending=False)
# si quisiera tambien tener en cuenta los ambientes
# hogares.groupby(['barrio','ambientes'])['precio_usd'].mean().sort_values(ascending=False)

barrio
alberdi           128280.100000
refineria         104000.000000
parque             92180.200000
hospitales         90100.000000
bella_vista        87000.500000
martin             80120.000000
san_martin         78000.000000
centro             76372.222222
saladillo          71666.666667
cinco_esquinas     67525.000000
echesortu          36500.000000
Name: precio_usd, dtype: float64

In [38]:
media_agrupada=hogares.groupby('barrio')['precio_usd'].transform('mean')
#transform lee y llena con precios promedios por barrio, al segundo depto de bella vista que aparezca le va a asignar el promedio y asi
# si hay un solo depto el valor va a ser el unico

In [39]:
data_grouped_mean=hogares.copy()

In [40]:
#Rellenamos el NA con la media agrupada
data_grouped_mean['precio_usd']=data_grouped_mean['precio_usd'].fillna(media_agrupada)

In [42]:
print(data_grouped_mean.iloc[[10,13]])

    id_propiedad distrito   barrio  ambientes  precio_usd
10            11   centro   martin          2     80120.0
13            14    norte  alberdi          2    128280.1
